### Homework 5: model deployment
#### Shuxiong Cai

Instructions: Using the F1 dataset, build a predictive model and log it in MLflow and write the ML model predictions into a database.

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import tempfile
import os
import mlflow
import mlflow.sklearn
from pyspark.sql import functions as F

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,ConfusionMatrixDisplay)
from sklearn.linear_model import LogisticRegression


#### Q1: Create two (2) new tables in your own fatabse where you'll store the predictions from each model for this exercise.

In [0]:
spark.sql("USE CATALOG gr5069")
spark.sql("CREATE DATABASE IF NOT EXISTS sc5816")


In [0]:
my_db = "gr5069.sc5816"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {my_db}.rf_podium_predictions (
    raceId INT,
    driverId INT,
    actual_label INT,
    predicted_label INT,
    predicted_probability DOUBLE,
    model_name STRING
)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {my_db}.lr_podium_predictions (
    raceId INT,
    driverId INT,
    actual_label INT,
    predicted_label INT,
    predicted_probability DOUBLE,
    model_name STRING
)
""")

In [0]:
spark.sql(f"SHOW TABLES IN {my_db}").show(truncate=False)


I created two new tables in my own database, `gr5069.sc5816`, to store the prediction outputs for the two models used in this assignment:
- `rf_podium_predictions`(Random Forest model)
- `lr_podium_predictions`(Logistic Regression model)

#### Q2: Build two (2) predictive models using MLflow, logging hyperparameters, the model itself, four metrics, and two artifcats. Submit submit your MLflow experiments as part of your assignments


I built two predictive models using MLflow:
- RandomForestClassifier
- LogisticRegression

For each run, I log the hyperparameters, the model itself, four metrics, and two artifacts.

In [0]:
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True, inferSchema=True)
df_races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True, inferSchema=True)
df_drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True, inferSchema=True)
df_qualifying = spark.read.csv("/Volumes/gr5069/raw/f1_data/qualifying.csv", header=True, inferSchema=True)

df_model = (
    df_results
    .select("raceId", "driverId", "constructorId", "positionOrder", "grid")
    .join(
        df_races.select("raceId", "year", "round", "circuitId", "date"),
        on="raceId",
        how="left"
    )
    .join(
        df_drivers.select("driverId", "dob"),
        on="driverId",
        how="left"
    )
    .join(
        df_qualifying
        .select("raceId", "driverId", "position")
        .withColumnRenamed("position", "quali_position"),
        on=["raceId", "driverId"],
        how="left"
    )
    .withColumn("label", F.when(F.col("positionOrder") <= 3, 1).otherwise(0))
    .withColumn("age", F.year("date") - F.year("dob"))
    .select(
        "raceId", "driverId", "label",
        "grid", "quali_position", "constructorId",
        "circuitId", "year", "round", "age"
    )
    .dropna()
)

display(df_model)

In [0]:
pdf = df_model.toPandas()

id_cols = pdf[["raceId", "driverId"]]
X = pdf[["grid", "quali_position", "constructorId", "circuitId", "year", "round", "age"]]
y = pdf["label"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y, id_cols,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [0]:
experiment_name = "/Users/sc5816@columbia.edu/HW5_model_deployment"
mlflow.set_experiment(experiment_name)

experiment = mlflow.get_experiment_by_name(experiment_name)
experimentID = experiment.experiment_id

print("Experiment ID:", experimentID)

In [0]:
rf_params = {
    "n_estimators": 100,
    "max_depth": 5,
    "random_state": 42
}

with mlflow.start_run(experiment_id=experimentID, run_name="HW5_RF_Model"):
    rf = RandomForestClassifier(**rf_params)
    rf.fit(X_train, y_train)
    rf_predictions = rf.predict(X_test)
    rf_probabilities = rf.predict_proba(X_test)[:, 1]

    mlflow.sklearn.log_model(rf, "random-forest-model")

    for param, value in rf_params.items():
        mlflow.log_param(param, value)

    rf_accuracy = accuracy_score(y_test, rf_predictions)
    rf_precision = precision_score(y_test, rf_predictions, zero_division=0)
    rf_recall = recall_score(y_test, rf_predictions, zero_division=0)
    rf_f1 = f1_score(y_test, rf_predictions, zero_division=0)

    mlflow.log_metric("accuracy", rf_accuracy)
    mlflow.log_metric("precision", rf_precision)
    mlflow.log_metric("recall", rf_recall)
    mlflow.log_metric("f1", rf_f1)

    rf_importance = pd.DataFrame({
        "Feature": X_train.columns,
        "Importance": rf.feature_importances_
    }).sort_values("Importance", ascending=False)

    temp = tempfile.NamedTemporaryFile(suffix=".csv", delete=False)
    rf_importance.to_csv(temp.name, index=False)
    mlflow.log_artifact(temp.name, "feature_importance")
    os.unlink(temp.name)

    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(y_test, rf_predictions, ax=ax)
    plt.title("RF Confusion Matrix")

    temp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    fig.savefig(temp.name, bbox_inches="tight")
    mlflow.log_artifact(temp.name, "plots")
    plt.close(fig)
    os.unlink(temp.name)

In [0]:
lr_params = {
    "C": 1.0,
    "max_iter": 1000,
    "random_state": 42
}

with mlflow.start_run(experiment_id=experimentID, run_name="HW5_LR_Model"):
    lr = LogisticRegression(**lr_params)
    lr.fit(X_train, y_train)
    lr_predictions = lr.predict(X_test)
    lr_probabilities = lr.predict_proba(X_test)[:, 1]

    mlflow.sklearn.log_model(lr, "logistic-regression-model")

    for param, value in lr_params.items():
        mlflow.log_param(param, value)

    lr_accuracy = accuracy_score(y_test, lr_predictions)
    lr_precision = precision_score(y_test, lr_predictions, zero_division=0)
    lr_recall = recall_score(y_test, lr_predictions, zero_division=0)
    lr_f1 = f1_score(y_test, lr_predictions, zero_division=0)

    mlflow.log_metric("accuracy", lr_accuracy)
    mlflow.log_metric("precision", lr_precision)
    mlflow.log_metric("recall", lr_recall)
    mlflow.log_metric("f1", lr_f1)

    lr_coef = pd.DataFrame({
        "Feature": X_train.columns,
        "Coefficient": lr.coef_[0]
    }).sort_values("Coefficient", ascending=False)

    temp = tempfile.NamedTemporaryFile(suffix=".csv", delete=False)
    lr_coef.to_csv(temp.name, index=False)
    mlflow.log_artifact(temp.name, "coefficients")
    os.unlink(temp.name)

    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(y_test, lr_predictions, ax=ax)
    plt.title("LR Confusion Matrix")

    temp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    fig.savefig(temp.name, bbox_inches="tight")
    mlflow.log_artifact(temp.name, "plots")
    plt.close(fig)
    os.unlink(temp.name)

#### Q3:  For each model, store its predictions in the corresponding table you created in your own database. Ensure you are using your own database to store your predictions.

In [0]:
rf_output = ids_test.copy()
rf_output["actual_label"] = y_test.values
rf_output["predicted_label"] = rf_predictions
rf_output["predicted_probability"] = rf_probabilities
rf_output["model_name"] = "RandomForestClassifier"

rf_output_spark = spark.createDataFrame(rf_output)

rf_output_spark.write.mode("overwrite").saveAsTable("gr5069.sc5816.rf_podium_predictions")

In [0]:
lr_output = ids_test.copy()
lr_output["actual_label"] = y_test.values
lr_output["predicted_label"] = lr_predictions
lr_output["predicted_probability"] = lr_probabilities
lr_output["model_name"] = "LogisticRegression"

lr_output_spark = spark.createDataFrame(lr_output)

lr_output_spark.write.mode("overwrite").saveAsTable("gr5069.sc5816.lr_podium_predictions")

In [0]:
spark.sql("SELECT * FROM gr5069.sc5816.rf_podium_predictions LIMIT 10").show()
spark.sql("SELECT * FROM gr5069.sc5816.lr_podium_predictions LIMIT 10").show()

In [0]:
spark.sql("SELECT COUNT(*) AS rf_rows FROM gr5069.sc5816.rf_podium_predictions").show()
spark.sql("SELECT COUNT(*) AS lr_rows FROM gr5069.sc5816.lr_podium_predictions").show()

#### Q4: Push your code to GitHub upon completion